In [1]:
%pip install \
    "qiskit==2.2.3" \
    "qiskit-machine-learning==0.9.0" \
    "qiskit-optimization==0.7.0" \
    "qiskit-algorithms==0.4.0" \
    "packaging>=24.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 11.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.7 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=f83cc776bd5f4c507b5d3285564e56b0b1154d8511ff75d842b77e453eb9b112
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [2]:
import sys
from importlib.metadata import version, requires
from packaging.requirements import Requirement
from packaging.markers import default_environment

import numpy as np
import scipy
import sklearn
import qiskit
import qiskit_algorithms
import qiskit_machine_learning
import qiskit_optimization
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

expected = {
    'qiskit': '2.2.3',
    'qiskit-machine-learning': '0.9.0',
    'qiskit-optimization': '0.7.0',
    'qiskit-algorithms': '0.4.0',
}
print('Python:', sys.version.split()[0])
environment = default_environment()
environment['extra'] = ''
for package, target in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == target, f'{package} 설치 셀을 다시 실행하세요.'
    for specification in requires(package) or []:
        requirement = Requirement(specification)
        if requirement.marker and not requirement.marker.evaluate(environment):
            continue
        actual = version(requirement.name)
        assert actual in requirement.specifier, f'의존성 확인: {specification}, 설치 {actual}'
for package in ('numpy', 'scipy', 'scikit-learn'):
    print(package, version(package))
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
probability = Statevector.from_instruction(qc).probabilities()
assert np.allclose(probability, [0.5, 0, 0, 0.5])
print('회로 확률:', probability)
print('패키지 의존성·import·회로 연산 확인 완료')

Python: 3.13.18
qiskit 2.2.3
qiskit-machine-learning 0.9.0
qiskit-optimization 0.7.0
qiskit-algorithms 0.4.0
numpy 2.4.6
scipy 1.16.3
scikit-learn 1.6.1
회로 확률: [0.5 0.  0.  0.5]
패키지 의존성·import·회로 연산 확인 완료


In [3]:
import numpy as np
from importlib.metadata import version
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('Qiskit:', version('qiskit'))
data = '1011'
assert set(data) <= {'0', '1'}
qc = QuantumCircuit(len(data))
for qubit, bit in enumerate(data[::-1]):
    if bit == '1':
        qc.x(qubit)
state = Statevector.from_instruction(qc)
print(qc.draw(output='text'))
print('측정 확률:', state.probabilities_dict())
assert np.isclose(state.probabilities()[int(data, 2)], 1.0)

Qiskit: 2.2.3
     ┌───┐
q_0: ┤ X ├
     ├───┤
q_1: ┤ X ├
     └───┘
q_2: ─────
     ┌───┐
q_3: ┤ X ├
     └───┘
측정 확률: {np.str_('1011'): np.float64(1.0)}


In [4]:
import numpy as np
from importlib.metadata import version
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('Qiskit:', version('qiskit'))
data = np.array([0.2, 0.4, 0.1, 0.3], dtype=float)
norm = np.linalg.norm(data)
assert norm > 0 and len(data) == 4
amplitudes = data / norm
qc = QuantumCircuit(2)
qc.initialize(amplitudes, [0, 1])
state = Statevector.from_instruction(qc)
print('L2 노름:', round(float(norm), 6))
print('진폭:', np.round(state.data, 6))
print('확률 00,01,10,11:', np.round(state.probabilities(), 6))
assert np.allclose(state.data, amplitudes)
assert np.isclose(state.probabilities().sum(), 1.0)

Qiskit: 2.2.3
L2 노름: 0.547723
진폭: [0.365148+0.j 0.730297+0.j 0.182574+0.j 0.547723+0.j]
확률 00,01,10,11: [0.133333 0.533333 0.033333 0.3     ]
